# 🌊 SonarGuard / NEMO — Google Colab Training Notebook

This notebook trains both stages of the NEMO marine debris computer vision pipeline on Google Colab's free GPU:
- **Stage 1 (Detector)**: RCDI-YOLO 1-Channel acoustic model on `SeabedObjects`
- **Stage 2 (Verifier)**: MobileNetV3-Small binary verifier on `Marine_PULSE`

> **GPU Setup**: Go to **Runtime → Change runtime type → T4 GPU** before starting.

### Step 1: Verify GPU Acceleration

In [ ]:
!nvidia-smi

### Step 2: Clone Repository & Install Dependencies
Replace `<YOUR_GITHUB_REPO_URL>` with your repository URL (or upload your repo zip file).

In [ ]:
# If using GitHub:
# !git clone <YOUR_GITHUB_REPO_URL> nemo
# %cd nemo

# If you uploaded a zip of NEMO to Colab:
# !unzip -q NEMO.zip -d nemo
# %cd nemo

!pip install -q -r requirements.txt
print("Environment ready!")

### Step 3: Prepare Datasets

Ensure your datasets are placed in:
- `data/raw/seabedobjects` (containing `train/`, `valid/`, `test/` or `images/` & `labels/`)
- `data/raw/marine_pulse` (containing category folders: `seabed surface`, `pipeline or cable`, etc.)

*(Optional: Mount Google Drive if your datasets are stored there)*

In [ ]:
# Optional: Mount Google Drive if your datasets are in Drive
# from google.colab import drive
# drive.mount('/content/drive')
# !mkdir -p data/raw/seabedobjects data/raw/marine_pulse
# !cp -r /content/drive/MyDrive/seabedobjects/* data/raw/seabedobjects/
# !cp -r /content/drive/MyDrive/marine_pulse/* data/raw/marine_pulse/

### Step 4: Train Stage 1 — RCDI-YOLO Candidate Detector
This trains the 1-channel acoustic YOLO model with LANConvNeXt backbone, DySample upsampler, and ImplicitHead.

In [ ]:
!python -m models.rcdi_yolo.train \
    --data data/raw/seabedobjects \
    --epochs 50 \
    --output models/rcdi_yolo/weights

### Step 5: Train Stage 2 — MobileNetV3 Verifier
This trains the lightweight verifier on sonar crops to distinguish artificial debris from natural seabed terrain.

In [ ]:
!python -m models.mobilenetv3.train \
    --data data/raw/marine_pulse \
    --epochs 30 \
    --output models/mobilenetv3/weights

### Step 6: Download Trained Weights to Your Local Machine
Run this cell to download both `.pt` checkpoint files directly to your browser download folder.

In [ ]:
from google.colab import files

# Download Stage 1 weights
files.download('models/rcdi_yolo/weights/rcdi_yolo_1c_best.pt')

# Download Stage 2 weights
files.download('models/mobilenetv3/weights/mobilenetv3_verifier_best.pt')

print("Weights downloaded! Copy them to your local project directory as shown in the README.")